## Initialisation

Ce notebook a pour objectif d'ingérer les données brutes dans la base PostgreSQL.

On commence par établir la connexion à la base, puis on charge les deux datasets sources :
- **releves_incidents** : journal des incidents machines (opérateur, sévérité, type de défaut, shift…)
- **telemetry** : relevés capteurs par machine (température, pression, tension, rotation, pièces produites)

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text, URL

DB_USER = "indusense_user"
DB_PASSWORD = "ThEP@ssW0rd"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "indusense_db"

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"✅ Connexion à PostgreSQL réussie ({DB_HOST}:{DB_PORT}/{DB_NAME})")
except Exception as e:
    print(f"❌ Échec de connexion : {e}")

df_incidents = pd.read_csv("artifacts/ingestions/incidents/releves_incidents.csv")
print(f"✅ releves_incidents.csv chargé : {df_incidents.shape[0]} lignes, {df_incidents.shape[1]} colonnes")

df_telemetry = pd.read_csv("artifacts/ingestions/incidents/telemetry.csv")
print(f"✅ telemetry.csv chargé : {df_telemetry.shape[0]} lignes, {df_telemetry.shape[1]} colonnes")

## Ingestion CSV → PostgreSQL

Insertion des deux datasets dans leurs tables bronze respectives. On utilise `if_exists="replace"` pour recréer les tables à chaque exécution.

In [ ]:
df_incidents["valid_parsing"] = True
df_telemetry["valid_parsing"] = True

try:
    df_incidents.to_sql("bronze_releves_incidents", engine, if_exists="replace", index=False)
    print(f"✅ {len(df_incidents)} lignes insérées dans 'bronze_releves_incidents'")
except Exception as e:
    print(f"❌ Échec pour 'bronze_releves_incidents' : {e}")

try:
    df_telemetry.to_sql("bronze_telemetry", engine, if_exists="replace", index=False)
    print(f"✅ {len(df_telemetry)} lignes insérées dans 'bronze_telemetry'")
except Exception as e:
    print(f"❌ Échec pour 'bronze_telemetry' : {e}")

## Validation — bronze_releves_incidents

Mise à jour de `valid_parsing` à `False` pour les lignes dont la valeur de `severity` est en dehors de l'intervalle [1, 5].

In [ ]:
from sqlalchemy import update, or_
from sqlalchemy.orm import Session
from models.bronze import BronzeRelevesIncidents

with Session(engine) as session:
    stmt = (
        update(BronzeRelevesIncidents)
        .where(
            or_(
                BronzeRelevesIncidents.severity < 1,
                BronzeRelevesIncidents.severity > 5,
            )
        )
        .values(valid_parsing=False)
    )
    result = session.execute(stmt)
    session.commit()
    print(f"✅ {result.rowcount} lignes marquées valid_parsing=False (severity hors [1, 5])")

## Validation — bronze_telemetry

Mise à jour de `valid_parsing` à `False` pour les lignes dont la combinaison `machine_id` + `timestamp` apparaît plus d'une fois (doublons).

In [18]:
from sqlalchemy import update, select, func, literal_column
from sqlalchemy.orm import Session
from models.bronze import BronzeTelemetry

ctid = literal_column('ctid')

with Session(engine) as session:
    ranked_subq = (
        select(
            ctid.label('row_ctid'),
            func.row_number().over(
                partition_by=[BronzeTelemetry.machine_id, BronzeTelemetry.timestamp],
                order_by=ctid
            ).label('rn')
        )
        .select_from(BronzeTelemetry)
        .subquery()
    )

    stmt = (
        update(BronzeTelemetry)
        .where(
            ctid.in_(
                select(ranked_subq.c.row_ctid).where(ranked_subq.c.rn > 1)
            )
        )
        .values(valid_parsing=False)
    )
    result = session.execute(stmt)
    session.commit()
    print(f"✅ {result.rowcount} lignes marquées valid_parsing=False (doublons - 2ème occurrence uniquement)")

✅ 1346 lignes marquées valid_parsing=False (doublons - 2ème occurrence uniquement)
